<a href="https://colab.research.google.com/github/atsuro0415/Deeplearning/blob/261017/lecture02_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第2回講義 宿題

## 課題
今回のLessonで学んだことを元に，MNISTのファッション版 (Fashion MNIST，クラス数10) をソフトマックス回帰によって分類してみましょう．

Fashion MNISTの詳細については以下のリンクを参考にしてください．

Fashion MNIST: https://github.com/zalandoresearch/fashion-mnist

### 目標値
Accuracy: 80%

### ルール
- 訓練データは`x_train`， `y_train`，テストデータは`x_test`で与えられます．
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- **下のセルで指定されている`x_train、y_train`以外の学習データは使わないでください．**
- **隠れ層のない単層のモデルとして実装してください.**
- **ソフトマックス回帰のアルゴリズム部分の実装はnumpyのみで行ってください** (sklearnやtensorflowなどは使用しないでください)．
    - データの前処理部分でsklearnの関数を使う (例えば `sklearn.model_selection.train_test_split`) のは問題ありません．

### 提出方法
- 2つのファイルを提出していただきます．
    1. テストデータ (`x_test`) に対する予測ラベルをcsv形式で保存し，**Omnicampusの宿題タブから「第2回 機械学習基礎」を選択して**提出してください．
    2. それに対応するpythonのコードを　ファイル＞ダウンロード＞.pyをダウンロード　から保存し，**Omnicampusの宿題タブから「第2回 機械学習基礎 (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコード全体をコピー&ペーストしてください．
      
- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）．コードの内容を変更した場合は，**1と2の両方を提出し直してください**．

### 評価方法
- 予測ラベルの`y_test`に対する精度 (Accuracy) で評価します．
- 即時採点しLeader Boardを更新します（採点スケジュールは別アナウンス）．
- 締切時の点数を最終的な評価とします．

### ドライブのマウント

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# 作業ディレクトリを指定
work_dir = '/content/drive/MyDrive/Colab_Notebooks/Deeplearning/第2回'

### データの読み込み（このセルは修正しないでください）

In [10]:
import os
import sys

import numpy as np
import pandas as pd

sys.modules['tensorflow'] = None

def load_fashionmnist():
    # 学習データ
    x_train = np.load(work_dir + '/Lecture02/data/x_train.npy')
    y_train = np.load(work_dir + '/Lecture02/data/y_train.npy')

    # テストデータ
    x_test = np.load(work_dir + '/Lecture02/data/x_test.npy')

    x_train = x_train.reshape(-1, 784).astype('float32') / 255
    y_train = np.eye(10)[y_train.astype('int32')]
    x_test = x_test.reshape(-1, 784).astype('float32') / 255

    return x_train, y_train, x_test

### ソフトマックス回帰の実装

In [20]:
x_train, y_train, x_test = load_fashionmnist()

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

def softmax(x):
    # 2次元の入力に対応するため、最期の次元(axis=-1)で最大値を引いてexpをとります
    x_max = x.max(axis=-1, keepdims=True)
    x_exp = np.exp(x - x_max)
    return x_exp / x_exp.sum(axis=-1, keepdims=True)

def np_log(x):
    # 0の対数を取るのを防ぐために極小値を足します
    return np.log(np.clip(x, 1e-15, 1.0))

# 重みの初期化
np.random.seed(42) # 再現性のため
W = np.random.normal(0, 0.01, (784, 10)) # 重み (784, 10)
b = np.zeros(10)                       # バイアス (10,)

# 学習データと検証データに分割
x_train, x_valid, y_train, y_valid = train_test_split(x_train, y_train, test_size=0.1, random_state=42)

def train(x, t, eps=0.1, batch_size=128):
    global W, b  # グローバル変数を更新するために明示的に宣言します

    # データのシャッフル
    indices = np.arange(x.shape[0])
    np.random.shuffle(indices)
    x_shuffled = x[indices]
    t_shuffled = t[indices]

    total_loss = 0
    num_batches = int(np.ceil(x.shape[0] / batch_size))

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min(start_idx + batch_size, x.shape[0])

        x_batch = x_shuffled[start_idx:end_idx]
        t_batch = t_shuffled[start_idx:end_idx]
        actual_batch_size = x_batch.shape[0]

        y_hat = softmax(np.matmul(x_batch, W) + b)
        cost = (- t_batch * np_log(y_hat)).sum(axis=1).mean()
        total_loss += cost * actual_batch_size

        delta = y_hat - t_batch  # (actual_batch_size, 出力の次元数)

        dW = np.matmul(x_batch.T, delta) / actual_batch_size  # (入力の次元数, 出力の次元数)
        db = np.matmul(np.ones(shape=(actual_batch_size,)), delta) / actual_batch_size  # (出力の次元数,)

        W -= eps * dW
        b -= eps * db

    return total_loss / x.shape[0]

def valid(x, t):
    y_hat = softmax(np.matmul(x, W) + b)
    cost = (- t * np_log(y_hat)).sum(axis=1).mean()

    return cost, y_hat  # 予測値（y_hat）も一緒に返します

# ミニバッチ学習を適用してエポックを回します
epochs = 30
batch_size = 128
learning_rate = 0.1

for epoch in range(epochs):
    cost_train = train(x_train, y_train, eps=learning_rate, batch_size=batch_size)
    cost_valid, y_pred_valid = valid(x_valid, y_valid)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print('EPOCH: {}, Train Cost: {:.3f}, Valid Cost: {:.3f}, Valid Accuracy: {:.3f}'.format(
            epoch + 1,
            cost_train,
            cost_valid,
            accuracy_score(y_valid.argmax(axis=1), y_pred_valid.argmax(axis=1))
        ))

# テストデータの予測
y_pred = softmax(np.matmul(x_test, W) + b)

submission = pd.Series(y_pred.argmax(axis=1), name='label')
submission.to_csv(work_dir + '/Lecture02/submission_pred_02.csv', header=True, index_label='id')

EPOCH: 1, Train Cost: 0.705, Valid Cost: 0.558, Valid Accuracy: 0.805
EPOCH: 5, Train Cost: 0.463, Valid Cost: 0.459, Valid Accuracy: 0.838
EPOCH: 10, Train Cost: 0.433, Valid Cost: 0.439, Valid Accuracy: 0.848
EPOCH: 15, Train Cost: 0.418, Valid Cost: 0.431, Valid Accuracy: 0.848
EPOCH: 20, Train Cost: 0.410, Valid Cost: 0.422, Valid Accuracy: 0.853
EPOCH: 25, Train Cost: 0.404, Valid Cost: 0.416, Valid Accuracy: 0.857
EPOCH: 30, Train Cost: 0.400, Valid Cost: 0.434, Valid Accuracy: 0.849
